# Đánh giá mô hình Academic Demo v2
Chạy các ô theo thứ tự trên Colab. Upload **academic-demo-v2-colab-20260917T081227Z-handoff.zip**.
Notebook phân tích lại các model đã fit trên **dev**, không train lại hoặc predict trên test. Final test chỉ được trình bày từ metrics đã lưu. Phân tích mới là hậu kiểm, không thay lựa chọn model đã đóng băng.

Mục tiêu: giải thích lựa chọn Logistic Regression, chất lượng đặc trưng, trade-off cảnh báo và giới hạn sử dụng. Dataset và event là synthetic; target là không hoàn thành học phần, không phải điểm số liên tục.


In [ ]:
import sys, subprocess
from importlib.metadata import version
PINS = {'numpy':'2.3.5','pandas':'2.3.3','scipy':'1.16.3','scikit-learn':'1.7.2','joblib':'1.5.2'}
assert sys.version_info[:2] == (3,13), 'Cần Python 3.13 giống môi trường artifact.'
missing=[]
for name,wanted in PINS.items():
    try: actual=version(name)
    except Exception: actual=None
    if actual != wanted: missing.append(f'{name}=={wanted}')
if missing:
    subprocess.run([sys.executable,'-m','pip','install',*missing],check=True)
    raise RuntimeError('Đã cài phiên bản đúng. Restart session rồi chạy lại từ ô này; không xóa runtime.')
print('Runtime tương thích. Dùng matplotlib của Colab để vẽ biểu đồ.')


## 1 Nhận handoff và xác minh trước khi nạp model
Chỉ chấp nhận đúng ZIP chủ dự án đã bàn giao. Hash kiểm tra cả nội dung, không chỉ tên file.

In [ ]:
from pathlib import Path
import zipfile, hashlib, json, io
EXPECTED_ZIP_SHA='30110b67d62f29003138a7c407a086139063ca5a3709aca5f101ee0fba60e8ee'
LOCAL_ZIP = None  # local: đặt đường dẫn handoff; Colab: giữ None
if LOCAL_ZIP is None:
    from google.colab import files
    uploaded=files.upload()
    assert len(uploaded)==1, 'Chọn một ZIP handoff'
    ZIP=Path(next(iter(uploaded)))
else: ZIP=Path(LOCAL_ZIP)
assert hashlib.sha256(ZIP.read_bytes()).hexdigest()==EXPECTED_ZIP_SHA, 'Sai ZIP hoặc nội dung đã thay đổi'
archive=zipfile.ZipFile(ZIP)
def read_json(name): return json.loads(archive.read(name))
def sha(name): return hashlib.sha256(archive.read(name)).hexdigest()
manifest=read_json('bundle/manifest.json')
receipt=read_json('activation_receipt.json')
assert sha('bundle/manifest.json')==receipt['manifest_sha256']
for name,meta in manifest['files'].items():
    assert sha('bundle/'+name)==meta['sha256']
    assert len(archive.read('bundle/'+name))==meta['size_bytes']
split_manifest=read_json('processed/splits_manifest.json')
for name,meta in split_manifest['files'].items():
    assert sha('processed/'+name+'.jsonl')==meta['sha256']
for name,wanted in manifest['environment']['packages'].items(): assert version(name)==wanted
print('PASS source ZIP, bundle, split hashes và package versions')
print('Model:',manifest['model_version'],'Source:',manifest['source_commit'])


## 2 Dữ liệu và EDA
Chỉ đọc bảng feature train/dev để phân tích. Test được kiểm tra hash nhưng không dùng cho EDA hay tuning. Giá trị thiếu được impute bằng median học từ train.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
FEATURES=manifest['feature_order']
def frame(split):
    rows=[json.loads(x) for x in archive.read('processed/'+split+'.jsonl').decode().splitlines()]
    X=pd.DataFrame([r['record']['features'] for r in rows],columns=FEATURES).astype(float)
    return X,np.array([r['target'] for r in rows]),np.array([r['group_id'] for r in rows]),[r['record']['case_id'] for r in rows]
Xtrain,ytrain,gtrain,ctrain=frame('train')
Xdev,ydev,gdev,cdev=frame('dev')
assert not set(gtrain)&set(gdev)
audit=read_json('audit/data_audit.json')
summary=pd.DataFrame([{ 'split':k,**v,'positive_prevalence':audit['positive_prevalence'][k]} for k,v in split_manifest['files'].items()]).drop(columns='sha256')
display(summary)
display(Xtrain.describe().T)
display(pd.DataFrame({'missing_train':Xtrain.isna().sum(),'missing_dev':Xdev.isna().sum()}))
fig,axs=plt.subplots(1,2,figsize=(11,4))
summary.plot.bar(x='split',y='row_count',ax=axs[0],legend=False,title='Số mẫu theo split')
summary.plot.bar(x='split',y='positive_prevalence',ax=axs[1],legend=False,title='Tỷ lệ nhãn dương')
plt.tight_layout(); plt.show()


In [ ]:
fig,axs=plt.subplots(2,4,figsize=(16,7))
for ax,feature in zip(axs.flat,FEATURES):
    for label in (0,1):
        values=Xtrain.loc[ytrain==label,feature].dropna()
        ax.hist(values,bins=20,alpha=.5,density=True,label=f'y={label}')
    ax.set_title(feature,fontsize=9); ax.legend()
axs.flat[-1].axis('off'); plt.tight_layout(); plt.show()
corr=Xtrain.corr()
fig,ax=plt.subplots(figsize=(9,7)); im=ax.imshow(corr,vmin=-1,vmax=1,cmap='coolwarm')
ax.set_xticks(range(len(FEATURES)),FEATURES,rotation=80); ax.set_yticks(range(len(FEATURES)),FEATURES)
fig.colorbar(im); ax.set_title('Tương quan feature trên train'); plt.tight_layout(); plt.show()


## 3 Vì sao chọn model
Chọn AP cao nhất trên dev theo protocol đã lưu, không chọn theo test. AP phù hợp dữ liệu hiếm nhãn dương; ROC-AUC không phải accuracy. So sánh cả F1, calibration và mức cảnh báo, nhưng không thay tiêu chí chọn sau khi xem test.


In [ ]:
import joblib
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve, roc_curve, confusion_matrix, precision_score, recall_score, f1_score, brier_score_loss
from sklearn.calibration import calibration_curve
comparison=pd.read_csv(io.BytesIO(archive.read('report_tables/model_comparison_dev.csv')))
models={}; probabilities={}
for row in comparison.to_dict('records'):
    name=row['model']; member='run/'+row['path']
    assert sha(member)==row['model_sha256']
    models[name]=joblib.load(io.BytesIO(archive.read(member)))
    probabilities[name]=models[name].predict_proba(Xdev)[:,list(models[name].classes_).index(1)]
    assert abs(average_precision_score(ydev,probabilities[name])-row['average_precision'])<1e-9
display(comparison.drop(columns=['path','model_sha256']))
selected=read_json('run/selected_run.json')['selected_model']
print('Đã chọn:',selected)
print('Toàn bộ tham số classifier và tiền xử lý đã lưu:')
for name,model in models.items(): print(name,model.get_params())
comparison.plot.bar(x='model',y=['average_precision','f1','roc_auc'],figsize=(10,4),title='Dev model comparison'); plt.tight_layout(); plt.show()


In [ ]:
fig,axs=plt.subplots(1,3,figsize=(16,4))
for name,p in probabilities.items():
    precision,recall,_=precision_recall_curve(ydev,p)
    axs[0].plot(recall,precision,label=name)
    fpr,tpr,_=roc_curve(ydev,p); axs[1].plot(fpr,tpr,label=name)
    frac,mean=calibration_curve(ydev,p,n_bins=10,strategy='quantile'); axs[2].plot(mean,frac,'o-',label=name)
axs[0].axhline(ydev.mean(),ls='--',color='gray'); axs[0].set(title='Dev Precision Recall',xlabel='Recall',ylabel='Precision')
axs[1].plot([0,1],[0,1],'--',color='gray'); axs[1].set(title='Dev ROC',xlabel='False positive rate',ylabel='True positive rate')
axs[2].plot([0,1],[0,1],'--',color='gray'); axs[2].set(title='Dev calibration',xlabel='Mean probability',ylabel='Observed fraction')
for ax in axs: ax.legend(fontsize=8)
plt.tight_layout(); plt.show()


## 4 Ngưỡng và chi phí cảnh báo
Đường cong này chỉ là hậu kiểm dev. Ngưỡng active vẫn là giá trị trong manifest. Không chọn ngưỡng mới theo các kết quả test đã biết.

In [ ]:
p=probabilities[selected]; threshold=manifest['alert_threshold']
thresholds=np.arange(.01,1,.01)
threshold_table=pd.DataFrame([{'threshold':t,'precision':precision_score(ydev,p>=t,zero_division=0),'recall':recall_score(ydev,p>=t),'f1':f1_score(ydev,p>=t),'alert_fraction':float(np.mean(p>=t))} for t in thresholds])
ax=threshold_table.plot(x='threshold',figsize=(10,4)); ax.axvline(threshold,color='black',ls='--',label='Active threshold'); ax.legend(); plt.show()
display(threshold_table.iloc[(threshold_table.threshold-threshold).abs().argsort()[:1]])
errors=pd.DataFrame({'case_id':cdev,'group_id':gdev,'target':ydev,'probability':p,'prediction':(p>=threshold).astype(int)})
errors['error_type']=np.select([(errors.target==0)&(errors.prediction==1),(errors.target==1)&(errors.prediction==0)],['false_positive','false_negative'],default='correct')
display(errors.groupby('error_type').size().rename('count'))
display(errors[errors.error_type=='false_positive'].sort_values('probability',ascending=False).head(10))
display(errors[errors.error_type=='false_negative'].sort_values('probability').head(10))
plt.hist(p[ydev==0],bins=30,alpha=.6,label='Actual 0'); plt.hist(p[ydev==1],bins=30,alpha=.6,label='Actual 1'); plt.axvline(threshold,color='black',ls='--'); plt.legend(); plt.title('Dev phân bố xác suất'); plt.show()


## 5 Đặc trưng ảnh hưởng như thế nào
Hệ số Logistic Regression sau StandardScaler là đóng góp theo đơn vị đã chuẩn hóa. Permutation importance đo mức giảm dev AP khi xáo một feature, không chứng minh quan hệ nhân quả. Feature tương quan có thể chia sẻ tầm quan trọng; attendance và missed sessions được sinh phụ thuộc nhau.


In [ ]:
from sklearn.inspection import permutation_importance
model=models[selected]
if hasattr(model.named_steps['classifier'],'coef_'):
    coef=pd.Series(model.named_steps['classifier'].coef_[0],index=FEATURES).sort_values()
    coef.plot.barh(title='Hệ số Logistic Regression đã chuẩn hóa',figsize=(9,4)); plt.tight_layout(); plt.show()
importance=permutation_importance(model,Xdev,ydev,scoring='average_precision',n_repeats=5,random_state=20260917,n_jobs=1)
imp=pd.DataFrame({'feature':FEATURES,'mean_AP_drop':importance.importances_mean,'std':importance.importances_std}).sort_values('mean_AP_drop')
display(imp)
plt.figure(figsize=(9,4)); plt.barh(imp.feature,imp.mean_AP_drop,xerr=imp['std']); plt.xlabel('Giảm AP dev'); plt.tight_layout(); plt.show()
if hasattr(model.named_steps['classifier'],'coef_'):
    index=int(np.argmax(p))
    contributions=model[:-1].transform(Xdev.iloc[[index]])[0]*model.named_steps['classifier'].coef_[0]
    display(pd.DataFrame({'feature':FEATURES,'raw_value':Xdev.iloc[index].values,'log_odds_contribution':contributions}))
    print('Case synthetic dev:',cdev[index], 'Probability:',p[index], 'Intercept:',model.named_steps['classifier'].intercept_[0])


## 6 Độ bất định trên dev
Bootstrap theo sinh viên bảo toàn sự phụ thuộc giữa nhiều lượt học của cùng một người. Khoảng percentile này là hậu kiểm trên model đã chọn, không bao gồm biến thiên do huấn luyện hoặc lựa chọn model và không phải CI của test.


In [ ]:
rng=np.random.default_rng(20260917); groups=np.unique(gdev); estimates=[]; differences=[]
indices={g:np.flatnonzero(gdev==g) for g in groups}
for _ in range(200):
    ix=np.concatenate([indices[g] for g in rng.choice(groups,len(groups),replace=True)])
    if len(np.unique(ydev[ix]))<2: continue
    ap=average_precision_score(ydev[ix],p[ix]); estimates.append(ap)
    differences.append(average_precision_score(ydev[ix],probabilities['logistic_regression'][ix])-average_precision_score(ydev[ix],probabilities['random_forest'][ix]))
print('Dev AP bootstrap 95%:',np.quantile(estimates,[.025,.975]))
print('Dev AP LR minus RF 95%:',np.quantile(differences,[.025,.975]))
plt.hist(estimates,bins=25); plt.title('Bootstrap AP theo nhóm sinh viên trên dev'); plt.show()


## 7 Final test đã đóng băng
Chỉ đọc JSON metrics. Không có predict trên test, không sinh ROC/PR test từ số liệu tổng hợp. Muốn đường ROC/PR test cần predictions lưu từ lần đánh giá đã phê duyệt; handoff hiện không chứa chúng.

In [ ]:
final=read_json('run/final_metrics.json')
display(pd.DataFrame([{'metric':k,'value':final[k]} for k in ['sample_count','positive_prevalence','average_precision','precision','recall','f1','roc_auc','brier','threshold']]))
fig,axs=plt.subplots(1,2,figsize=(11,4)); cm=np.array(final['confusion_matrix'])
axs[0].imshow(cm,cmap='Blues'); axs[0].set(xticks=[0,1],yticks=[0,1],xlabel='Predicted',ylabel='Actual',title='Final confusion matrix')
for i in range(2):
    for j in range(2): axs[0].text(j,i,str(cm[i,j]),ha='center',color='white' if cm[i,j]>1500 else 'black')
c=final['calibration']; axs[1].plot(c['mean_predicted_probability'],c['fraction_positive'],'o-'); axs[1].plot([0,.3],[0,.3],'--'); axs[1].set(title='Final calibration đã lưu',xlabel='Mean probability',ylabel='Observed fraction'); plt.tight_layout(); plt.show()
tn,fp,fn,tp=cm.ravel()
print(f'{tp+fp} cảnh báo: {tp} đúng, {fp} sai; bỏ sót {fn}/{tp+fn} ca dương.')
print('AP / test prevalence (tham chiếu ngẫu nhiên):',final['average_precision']/final['positive_prevalence'])


## 8 Giới hạn huấn luyện và điều kiện áp dụng

1. Event là proxy sinh bằng RNG từ lịch sử, không phải dữ liệu LMS quan sát. Các quan hệ trong biểu đồ phản ánh cả công thức sinh dữ liệu.
2. Builder tích lũy prior theo semester/attempt/course, chưa lọc finalized_at trước cutoff; có nguy cơ rò rỉ điểm cùng học kỳ. Group split không khắc phục rò rỉ thời gian này.
3. prior_gpa_4 chưa chắc trùng GPA học vụ theo chính sách thay thế lần học. 145 lượt thiếu timestamp vẫn là giới hạn temporal audit.
4. Chỉ một bộ dữ liệu, một seed, chưa external validation hoặc đánh giá theo khóa tương lai; không kết luận tổng quát hóa thực tế.
5. Mô hình chọn theo AP, threshold chọn F1 trên dev, chưa tối ưu chi phí cố vấn hoặc chi phí bỏ sót. Calibration chưa hiệu chỉnh riêng.
6. Final test đã được xem trong lịch sử dự án. Phân tích này là hậu kiểm, không tạo bằng chứng test độc lập mới.
7. source_worktree_dirty=true được giữ trong artifact; runner dùng fallback này khi không có Git. Không suy ra trạng thái Git sạch từ trường đó.

Trước ứng dụng thực: event/timestamp hợp lệ, feature trước cutoff, target thống nhất, dữ liệu được phép sử dụng, split nhóm và thời gian, protocol mới, đánh giá ngoài mẫu và thử nghiệm có giám sát. Không tự đổi artifact active từ notebook này.

## 9 Đối chiếu Capstone
Data selection/transformation: ô 1–2; methods/model choice: ô 3–4; insight/visualization: ô 2–7; limits: ô 8. P6.2 có metrics, confusion matrix, ví dụ lỗi **dev** và limitations. Ví dụ lỗi test, benchmark RAG, browser E2E, clean-clone, slide/video vẫn là các hạng mục riêng. Hệ số và permutation importance không phải bằng chứng tác động nhân quả.


In [ ]:
# Xuất các bảng hậu kiểm vào thư mục mới; không sửa run gốc.
from datetime import datetime, timezone
OUT=Path('evaluation-report-'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
OUT.mkdir(exist_ok=False)
comparison.to_csv(OUT/'dev_model_comparison.csv',index=False)
threshold_table.to_csv(OUT/'dev_threshold_analysis.csv',index=False)
imp.to_csv(OUT/'dev_permutation_importance.csv',index=False)
errors.to_csv(OUT/'dev_error_analysis.csv',index=False)
(OUT/'analysis_protocol.json').write_text(json.dumps({'source_archive_sha256':EXPECTED_ZIP_SHA,'model_version':manifest['model_version'],'analysis':'posthoc_dev_only','test_recomputed':False,'bootstrap_repeats':200,'permutation_repeats':5},indent=2))
print('Đã lưu:',OUT.resolve())
# Colab: File > Download > Download .ipynb để lưu toàn bộ biểu đồ/output sau khi chạy.
